# Data Cleaning
This notebook is intended to demonstrate what can be done with scraped and converted JSON files retrieved in Step1, Step2, and Step3 notebooks. In this notebook, I create and clean dataframes from already-saved JSON files in 'data/json_files/candela_json_files', in preparation for creating charts to help analyze the fanwork metadata in visualizations.ipynb.

Specifically, this notebook:
1) Reads in a list of file paths to JSON files for individual pagination links,
2) Creates dataframes for each JSON file and concatenates the dataframes into a single dataframe,
3) Cleans the merged dataframe in preparation for data visualization,
4) Creates "exploded" dataframes in preparation for data visualization (for columns containing entries with more than one value in them, e.g. 'Category,' Archive Warnings', Characters,' 'Relationships,' or 'Additional Tags'),
5) Saves the cleaned dataframes to csv files for safekeeping.

In [1]:
# import libraries
import pandas as pd
import dateparser

# Get Combined Dataframe

In [2]:
def get_combined_df(filepath_list):
    """Gets combined dataframe from a list of filepaths."""
    # set dataframe list
    frames_list = []

    # convert JSON files to dataframe
    for i in range(len(filepath_list)):
        df = pd.read_json(filepath_list[i])
        df = df.T
        frames_list.append(df)

    # concatenate dataframes to overall df
    overall_df = pd.concat(frames_list)
    overall_df = overall_df.reset_index()
    return overall_df

# Cleaning: single-value columns
(Includes: 'Rating', 'Language', 'Date Published', 'Date Updated', 'Word Count', 'Chapter Count', 'Comment Count', 'Kudos Count', and 'Hits Count')

In [15]:
def parse_date_published(df):
    """Parses strings in 'Date Published' column and converts them to date objects."""
    for i in range(len(df)):
        current_date_published = df['Date Published'][i]
        if current_date_published != 'Unknown' and type(current_date_published) == str:
            date = dateparser.parse(current_date_published)
            df['Date Published'] = df['Date Published'].replace(current_date_published, date)
        else:
            pass
    return df

In [16]:
def parse_date_updated(df):
    """Parses strings in 'Date Updated' column and converts them to date objects."""
    for i in range(len(df)):
        current_date_updated = df['Date Updated'][i]
        if current_date_updated != 'Unknown' and type(current_date_updated) == str:
            date = dateparser.parse(current_date_updated)
            df['Date Updated'] = df['Date Updated'].replace(current_date_updated, date)
        else:
            pass
    return df

In [27]:
def fill_date_updated(df):
    """Fills in 'Date Updated' column if chapter count = 1/1
    (implies that Date Updated = Date Published).
    """
    # fill in date_updated based on chapter count
    for i in range(len(df)):
        chapter_count_current = df['Chapter Count'][i]
        current_date_updated = df['Date Updated'][i]
        current_date_published = df['Date Published'][i]
        if chapter_count_current == '1/1' and current_date_updated == 'Unknown':
            df.iloc[i, 10] = current_date_published
        else:
            pass
    return df

In [57]:
def update_language(df):
    """Fixes error in with language column from scraping non-adult fanworks."""
    lang_string = df['Language'][1]
    for i in range(len(df)):
        if df['Language'][i] == lang_string:
            lang_list = lang_string.split('\n')
            language = lang_list[1]
            language = language.strip()
            df.iloc[i, 8] = language
        else:
            continue
    return df

In [70]:
def clean_wordcount(df):
    """In 'Word Count' column, detects any strings and removes commas."""
    for i in range(len(df)):
        current_word_count = df['Word Count'][i]
        if type(current_word_count) == str:
            df.iloc[i, 11] = current_word_count.replace(",","")
        else:
            pass
    return df

In [66]:
def clean_comments(df):
    """In 'Comment Count' column, detects any strings and removes commas."""
    for i in range(len(df)):
        current_comment_count = df['Comment Count'][i]
        if type(current_comment_count) == str:
            df.iloc[i, 13] = current_comment_count.replace(",","")
        else:
            pass
    return df

In [68]:
def clean_kudos(df):
    """In 'Kudos Count' column, detects any strings and removes commas."""
    for i in range(len(df)):
        current_kudos_count = df['Kudos Count'][i]
        if type(current_kudos_count) == str:
            df.iloc[i, 14] = current_kudos_count.replace(",","")
        else:
            pass
    return df

In [72]:
def clean_bookmarks(df):
    """In 'Bookmarks Count' column, detects any strings and removes commas."""
    for i in range(len(df)):
        bookmarks_count = df['Bookmarks Count'][i]
        if type(bookmarks_count) == str:
            df.iloc[i, 15] = bookmarks_count.replace(",","")
        else:
            pass
    return df

In [62]:
def clean_hits(df):
    """In 'Hits Count' column, detects any strings and removes commas."""
    for i in range(len(df)):
        current_hits_count = df['Hits Count'][i]
        if type(current_hits_count) == str:
            df.iloc[i, 16] = current_hits_count.replace(",","")
        else:
            pass
    return df

In [74]:
def to_numeric(df, cols_list):
    """Takes in a dataframe and list of numeric columns and converts columns to numeric (in case there are any sneaky strings lurking."""
    df[cols_list] = df[cols_list].apply(pd.to_numeric)
    return df

In [94]:
def save_df_to_csv(filepath, df):
    """Saves dataframe to csv file."""
    return df.to_csv(filepath)

# Testing - Get combined dataframe

In [3]:
# set filepath list
filepath_list = ['/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page1.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page2.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page3.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page4.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page5.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page7.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page8.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page9.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page10.json',
                 '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_json_files/candela_page11.json']

In [17]:
candela_df = get_combined_df(filepath_list)

In [18]:
display(candela_df)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",\n English\n,2024-03-07,Unknown,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",\n English\n,2024-03-02,Unknown,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,27 Feb 2024,8013,None,3,10,3,160
3,4,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,27 Feb 2024,8013,None,3,10,3,160
4,5,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Elsie Roberts (Candela Obscura), Oscar Grimm,...","[Rajan Savarimuthu/Original character, Former ...","[Murder Mystery, Arson, Disability, Canon Disa...",\n English\n,2024-02-25,2024-02-25,46890,28/28,28,7,2,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],\n English\n,2023-07-04,Unknown,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,03 Jul 2023,6125,None,9,83,8,"1,259"
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",\n English\n,2023-07-02,Unknown,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",\n English\n,2023-06-26,Unknown,1082,1/1,2,27,4,220


In [19]:
candela_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             211 non-null    int64 
 1   Rating            211 non-null    object
 2   Archive Warnings  211 non-null    object
 3   Category          211 non-null    object
 4   Fandom            211 non-null    object
 5   Characters        211 non-null    object
 6   Relationships     211 non-null    object
 7   Additional Tags   211 non-null    object
 8   Language          211 non-null    object
 9   Date Published    211 non-null    object
 10  Date Updated      211 non-null    object
 11  Word Count        211 non-null    object
 12  Chapter Count     211 non-null    object
 13  Comment Count     211 non-null    object
 14  Kudos Count       211 non-null    object
 15  Bookmarks Count   211 non-null    object
 16  Hits Count        211 non-null    object
dtypes: int64(1), object(16)
mem

# Testing - clean single-value cols

## clean date columns

In [20]:
# get 'Unknown' count for 'Date Published' and 'Date Updated'
count_unknown_pub = candela_df['Date Published'].value_counts().get('Unknown', 0)
print("Number of rows where 'Date Published' == 'Unknown':")
print(count_unknown_pub)

count_unknown_update = candela_df['Date Updated'].value_counts().get('Unknown', 0)
print("Number of rows where 'Date Updated' == 'Unknown':")
print(count_unknown_update)

Number of rows where 'Date Published' == 'Unknown':
114
Number of rows where 'Date Updated' == 'Unknown':
77


In [22]:
# parse date strings / convert to date objects
candela_df = parse_date_published(candela_df)
candela_df = parse_date_updated(candela_df)

In [28]:
# for any rows with Date Published = 'Unknown' and 1/1 chapters, fill in with Date Updated
candela_df = fill_date_updated(candela_df)

In [29]:
display(candela_df)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",\n English\n,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",\n English\n,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
3,4,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
4,5,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Elsie Roberts (Candela Obscura), Oscar Grimm,...","[Rajan Savarimuthu/Original character, Former ...","[Murder Mystery, Arson, Disability, Canon Disa...",\n English\n,2024-02-25 00:00:00,2024-02-25 00:00:00,46890,28/28,28,7,2,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],\n English\n,2023-07-04 00:00:00,2023-07-04 00:00:00,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,"1,259"
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",\n English\n,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",\n English\n,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


## clean language

In [58]:
candela_df = update_language(candela_df)

In [59]:
# eyeballing changes
display(candela_df)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
3,4,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
4,5,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Elsie Roberts (Candela Obscura), Oscar Grimm,...","[Rajan Savarimuthu/Original character, Former ...","[Murder Mystery, Arson, Disability, Canon Disa...",English,2024-02-25 00:00:00,2024-02-25 00:00:00,46890,28/28,28,7,2,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],English,2023-07-04 00:00:00,2023-07-04 00:00:00,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,"1,259"
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


## clean numeric cols

In [77]:
candela_df = clean_wordcount(candela_df)
candela_df = clean_comments(candela_df)
candela_df = clean_kudos(candela_df)
candela_df = clean_bookmarks(candela_df)
candela_df = clean_hits(candela_df)

In [78]:
numeric_cols = ['Kudos Count', 'Bookmarks Count', 'Hits Count', 'Comment Count', 'Word Count']
candela_df = to_numeric(candela_df, numeric_cols)

In [79]:
# verify changes
candela_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             211 non-null    int64 
 1   Rating            211 non-null    object
 2   Archive Warnings  211 non-null    object
 3   Category          211 non-null    object
 4   Fandom            211 non-null    object
 5   Characters        211 non-null    object
 6   Relationships     211 non-null    object
 7   Additional Tags   211 non-null    object
 8   Language          211 non-null    object
 9   Date Published    211 non-null    object
 10  Date Updated      211 non-null    object
 11  Word Count        211 non-null    int64 
 12  Chapter Count     211 non-null    object
 13  Comment Count     211 non-null    int64 
 14  Kudos Count       211 non-null    int64 
 15  Bookmarks Count   211 non-null    int64 
 16  Hits Count        211 non-null    int64 
dtypes: int64(6), object(11)
mem

## verify ratings

In [80]:
unique_ratings = list(candela_df['Rating'].unique())
print(unique_ratings)
# Ratings list has all unique values; no changes needed

['Teen And Up Audiences', 'General Audiences', 'Explicit', 'Mature', 'Not Rated']


In [95]:
save_df_to_csv('/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/cleaned_candela_df.csv', candela_df)

# Create exploded dataframes

In [82]:
df_characters = candela_df.explode("Characters")
df_relationships = candela_df.explode("Relationships")
df_fandom = candela_df.explode("Fandom")
df_additional_tags = candela_df.explode("Additional Tags")
df_categories = candela_df.explode("Category")

In [147]:
df_warnings = candela_df.explode("Archive Warnings")

# Clean exploded dataframes

In [91]:
def clean_df_characters(df):
    """Simplifies text in dataframe 'Characters' column."""
    # replace text
    character_text_replace = {'Original':'Original Characters',
                          'Original Female Character(s)':'Original Characters',
                          'Original Male Character(s)':'Original Characters',
                          'Sean Finnerty (Candela Obscura)':'Sean Finnerty',
                          'Jimmy Finnerty (Candela Obscura)':'Jimmy Finnerty',
                          'Elsie Roberts (Candela Obscura)':'Elsie Roberts',
                          'Howard Margrove (Candela Obscura)':'Howard Margrove',
                          'Arlo Black (Candela Obscura)':'Arlo Black',
                          'Tony Finnerty (Candela Obscura)':'Tony Finnerty',
                          'Godot (Candela Obscura)':'Godot',
                          'Cosmo Grimm (Candela Obscura)':'Cosmo Grimm',
                          'Margaret "Peggy" Finnerty (Candela Obscura)':'Margaret "Peggy" Finnerty',
                          "Alexandra Elise O'Neill (Candela Obscura)":"Alexandra Elise O'Neill",
                          'lucas (Candela Obscura)':'Lucas Suarez',
                          'Jinnah "Jean" Basar (Briefly)':'Dr. Jinnah "Jean" Basar',
                          'Nathaniel Trapp (Mentioned)':'Nathaniel Trapp',
                          'August James (Candela Obscura)':'August "Auggie" James',
                          'Leo Amicus (Mentioned)':'Leo Amicus',
                          'Edgar Lycoris (Mentioned)':'Edgar Lycoris',
                          "Godot | Cosmo Grimm's Dog (Candela Obscura)":'Godot',
                          'Mina (Candela Obscura)':'Mina',
                          'Iomene (Candela Obscura)':'Iomene',
                          'Rajan Savarimuthu (Mentioned)':'Rajan Savarimuthu',
                          'Cordelia Glask (Mentioned)':'Cordelia Glask',
                          'Nokari (mentioned)':'Nokari',
                          'Mentioned Grimoria (Candela Obscura)':'Grimoria',
                          'Mentioned Malcolm Trills':'Malcolm Trills',
                          'Mentioned Decklan Murphy (Candela Obscura)':'Decklan Murphy',
                          'Felix (Candela Obscura)':'Felix',
                          'Jinnah "Jean" Basar':'Dr. Jinnah "Jean" Basar',
                          'Alexandra (Candela Obscura)':"Alexandra Elise O'Neill",
                          'Eddie(mentioned)':'Eddie',
                          'Dean (Candela Obscura)':'Dean',
                          "Arlo Black's Father (Candela Obscura)":"Arlo Black's Father",
                          'Decklan Murphy (Mentioned)':'Decklan Murphy'
                          }

    for key in character_text_replace.keys():
        df['Characters'] = df['Characters'].replace(key, character_text_replace[key])
    return df

In [99]:
def clean_df_relationships(df):
    """Simplifies text in dataframe 'Relationships' column."""
    relationship_text_replace = {'Sean Finnerty (Candela Obscura) & Original Character(s)':'Sean Finnerty & Original Character(s)',
                             'Sean Finnerty (Candela Obscura)/Original Female Character(s)':'Sean Finnerty/Original Female Character(s)',
                             'Sean Finnerty (Candela Obscura)/Original Character(s)':'Sean Finnerty/Original Character(s)',
                             'Jimmy Finnerty & Sean Finnerty (Candela Obscura)':'Jimmy Finnerty & Sean Finnerty',
                             'Rajan Savarimuthu/Original character':'Rajan Savarimuthu/Original Character(s)',
                             'Mina & Elsie Roberts (Candela Obscura)':'Mina & Elsie Roberts',
                             'Implied Sean Finnerty/Marion Collodi':'Sean Finnerty/Marion Collodi',
                             'Jean Basar/Sean Finnerty/Marion Collodi':'Sean Finnerty/Marion Collodi/Jinnah "Jean" Basar',
                             'Sean Finnerty/Marion Collodi (implied)':'Sean Finnerty/Marion Collodi',
                             'Howard Margrove & Dean (Candela Obscura)':'Howard Margrove & Dean',
                             'August "Auggie" James/ Arlo Black':'Arlo Black/August "Auggie" James'
                          }

    for key in relationship_text_replace.keys():
        df['Relationships'] = df['Relationships'].replace(key, relationship_text_replace[key])
    return df

In [105]:
def clean_df_fandom(df):
    """Simplifies text in dataframe 'Fandom' column."""
    fandom_text_replace = {'Candela Obscura (Critical Role Web Series)':'Candela Obscura',
                           'Critical Role (Web Series)':'Candela Obscura'}

    for key in fandom_text_replace.keys():
        df['Fandom'] = df['Fandom'].replace(key, fandom_text_replace[key])
    return df

# Test - Clean exploded dataframes

## clean df_characters

In [83]:
df_characters.info()

<class 'pandas.DataFrame'>
Index: 801 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             801 non-null    int64 
 1   Rating            801 non-null    object
 2   Archive Warnings  801 non-null    object
 3   Category          801 non-null    object
 4   Fandom            801 non-null    object
 5   Characters        801 non-null    str   
 6   Relationships     801 non-null    object
 7   Additional Tags   801 non-null    object
 8   Language          801 non-null    object
 9   Date Published    801 non-null    object
 10  Date Updated      801 non-null    object
 11  Word Count        801 non-null    int64 
 12  Chapter Count     801 non-null    object
 13  Comment Count     801 non-null    int64 
 14  Kudos Count       801 non-null    int64 
 15  Bookmarks Count   801 non-null    int64 
 16  Hits Count        801 non-null    int64 
dtypes: int64(6), object(10), str(1)


In [84]:
display(df_characters)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),Malcolm Trills,Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),Leo Amicus,Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),Grimoria (Candela Obscura),Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),Edgar Lycoris,Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),Grimoria,Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),Arlo Black (Candela Obscura),"Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"August ""Auggie"" James","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),Arlo Black (Candela Obscura),"[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),Howard Margrove (Candela Obscura),"[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [92]:
df_characters = clean_df_characters(df_characters)

In [93]:
unique_characters = list(df_characters['Characters'].unique())
print(unique_characters)

['Malcolm Trills', 'Leo Amicus', 'Grimoria (Candela Obscura)', 'Edgar Lycoris', 'Grimoria', 'Decklan Murphy', 'Sean Finnerty', 'Marion Collodi', 'Nathaniel Trapp', 'Dr. Jinnah "Jean" Basar', 'Beatrix Monroe | Auntie Bee', 'Jimmy Finnerty', 'Original Characters', 'Elsie Roberts', 'Oscar Grimm', 'Cosmo Grimm', 'Cordelia Glask', 'Godot', 'Minor Characters', 'Magpie - Character', 'Rajan Savarimuthu', 'Mina', 'Iomene', '(mentioned)', 'Amadeus Porthos (OMC)', 'The Mother (Candela Obscura)', 'Zora Manning', 'Nokari', 'rpg character - Character', 'Felix', 'August "Auggie" James', 'Arlo Black', 'Howard Margrove', 'Charlotte Eaves', "Alexandra Elise O'Neill", 'Eddie Easton', 'The Foggs', 'Lucas Suarez', 'Margaret "Peggy" Finnerty', 'Tony Finnerty', 'Violet Boucher', 'ish - Character', 'Field Officer Aaron Weimer', "Arlo's Father", "Arlo Black's Father", 'Eddie', 'Dean', 'Charlotte and Howard are only mentioned']


In [96]:
# save df_characters to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/character_df_exploded.csv'
save_df_to_csv(path_to_save, df_characters)

## clean df_relationships

In [97]:
df_relationships.info()

<class 'pandas.DataFrame'>
Index: 353 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             353 non-null    int64 
 1   Rating            353 non-null    object
 2   Archive Warnings  353 non-null    object
 3   Category          353 non-null    object
 4   Fandom            353 non-null    object
 5   Characters        353 non-null    object
 6   Relationships     353 non-null    str   
 7   Additional Tags   353 non-null    object
 8   Language          353 non-null    object
 9   Date Published    353 non-null    object
 10  Date Updated      353 non-null    object
 11  Word Count        353 non-null    int64 
 12  Chapter Count     353 non-null    object
 13  Comment Count     353 non-null    int64 
 14  Kudos Count       353 non-null    int64 
 15  Bookmarks Count   353 non-null    int64 
 16  Hits Count        353 non-null    int64 
dtypes: int64(6), object(10), str(1)


In [98]:
display(df_relationships)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",Sean Finnerty (Candela Obscura) & Original Cha...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",Sean Finnerty (Candela Obscura)/Original Femal...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",Sean Finnerty (Candela Obscura)/Original Chara...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,1259
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...",Arlo Black & Howard Margrove,"[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...",Arlo Black/Howard Margrove,"[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [100]:
df_relationships = clean_df_relationships(df_relationships)

In [101]:
unique_relationships = list(df_relationships['Relationships'].unique())
print(unique_relationships)

['Leo Amicus & Malcolm Trills', 'Grimoria & Malcolm Trills', 'Sean Finnerty & Original Character(s)', 'Sean Finnerty/Original Female Character(s)', 'Sean Finnerty/Original Character(s)', 'Marion Collodi & Sean Finnerty', 'Marion Collodi & Sean Finnerty & Nathaniel Trapp', 'Jinnah "Jean" Basar & Sean Finnerty', 'Sean Finnerty & Beatrix Monroe | Auntie Bee', 'Sean Finnerty & Nathaniel Trapp', 'Jimmy Finnerty & Sean Finnerty', 'Rajan Savarimuthu/Original Character(s)', 'Former Rajan/Elsie', 'Elsie Roberts/Rajan Savarimuthu', 'Oscar Grimm/Rajan Savarimuthu', 'No relationships tagged', 'Marion Collodi & Nathaniel Trapp', 'Marion Collodi/Nathaniel Trapp', 'Elsie Roberts & Mina', 'Elsie Roberts/Rajan Savarimuthu (Implied)', 'Sean Finnerty/Nathaniel Trapp', 'Marion Collodi/Sean Finnerty/Nathaniel Trapp', 'Marion Collodi/Sean Finnerty', 'Jinnah "Jean" Basar/Marion Collodi/Sean Finnerty', 'No Romantic Relationship(s)', 'Leo Amicus & Grimoria', 'Leo Amicus & Edgar Lycoris', 'Grimoria & Edgar Lyco

In [102]:
# save df_relationships to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/relationships_df_exploded.csv'
save_df_to_csv(path_to_save, df_relationships)

## clean df_fandom

In [103]:
df_fandom.info()

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             211 non-null    int64 
 1   Rating            211 non-null    object
 2   Archive Warnings  211 non-null    object
 3   Category          211 non-null    object
 4   Fandom            211 non-null    str   
 5   Characters        211 non-null    object
 6   Relationships     211 non-null    object
 7   Additional Tags   211 non-null    object
 8   Language          211 non-null    object
 9   Date Published    211 non-null    object
 10  Date Updated      211 non-null    object
 11  Word Count        211 non-null    int64 
 12  Chapter Count     211 non-null    object
 13  Comment Count     211 non-null    int64 
 14  Kudos Count       211 non-null    int64 
 15  Bookmarks Count   211 non-null    int64 
 16  Hits Count        211 non-null    int64 
dtypes: int64(6), object(10), st

In [104]:
display(df_fandom)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
3,4,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
4,5,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Elsie Roberts (Candela Obscura), Oscar Grimm,...","[Rajan Savarimuthu/Original character, Former ...","[Murder Mystery, Arson, Disability, Canon Disa...",English,2024-02-25 00:00:00,2024-02-25 00:00:00,46890,28/28,28,7,2,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],English,2023-07-04 00:00:00,2023-07-04 00:00:00,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,1259
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [106]:
unique_fandoms = list(df_fandom['Fandom'].unique())
print(unique_fandoms)

['Candela Obscura (Critical Role Web Series)', 'Critical Role (Web Series)', 'Candela Obscura']


In [107]:
df_fandom = clean_df_fandom(df_fandom)

In [108]:
unique_fandoms = list(df_fandom['Fandom'].unique())
print(unique_fandoms)

['Candela Obscura']


In [109]:
# save df_fandom to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/fandom_df_exploded.csv'
save_df_to_csv(path_to_save, df_fandom)

## clean df_warnings

In [148]:
df_warnings.info()

<class 'pandas.DataFrame'>
Index: 235 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             235 non-null    int64 
 1   Rating            235 non-null    object
 2   Archive Warnings  235 non-null    str   
 3   Category          235 non-null    object
 4   Fandom            235 non-null    object
 5   Characters        235 non-null    object
 6   Relationships     235 non-null    object
 7   Additional Tags   235 non-null    object
 8   Language          235 non-null    object
 9   Date Published    235 non-null    object
 10  Date Updated      235 non-null    object
 11  Word Count        235 non-null    int64 
 12  Chapter Count     235 non-null    object
 13  Comment Count     235 non-null    int64 
 14  Kudos Count       235 non-null    int64 
 15  Bookmarks Count   235 non-null    int64 
 16  Hits Count        235 non-null    int64 
dtypes: int64(6), object(10), str(1)


In [149]:
display(df_warnings)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"Graphic Depictions Of Violence, Major Characte...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
2,3,Explicit,Graphic Depictions Of Violence,F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
2,3,Explicit,Major Character Death,F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],English,2023-07-04 00:00:00,2023-07-04 00:00:00,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,1259
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [150]:
unique_warnings = list(df_warnings['Archive Warnings'].unique())
print(unique_warnings)

['No Archive Warnings Apply', 'Graphic Depictions Of Violence, Major Character Death', 'Graphic Depictions Of Violence', 'Major Character Death', 'Creator Chose Not To Use Archive Warnings', 'Creator Chose Not To Use Archive Warnings, No Archive Warnings Apply']


In [151]:
# remove any rows where there's more than one value
text_to_remove = ['Graphic Depictions Of Violence, Major Character Death','Creator Chose Not To Use Archive Warnings, No Archive Warnings Apply']

In [152]:
indices_to_drop = df_warnings[df_warnings['Archive Warnings'] == 'Graphic Depictions Of Violence, Major Character Death'].index
indices_to_drop = list(indices_to_drop)
print(indices_to_drop)

[2, 3, 50, 51, 73, 74, 96, 97, 119, 120, 153]


In [153]:
indices_to_drop1 = df_warnings[df_warnings['Archive Warnings'] == 'Creator Chose Not To Use Archive Warnings, No Archive Warnings Apply'].index
indices_to_drop1 = list(indices_to_drop1)
print(indices_to_drop1)

[26]


In [154]:
indices_to_drop.extend(indices_to_drop1)

In [155]:
print(indices_to_drop)

[2, 3, 50, 51, 73, 74, 96, 97, 119, 120, 153, 26]


In [156]:
df_warnings.drop(indices_to_drop, inplace=True)

In [157]:
unique_warnings = list(df_warnings['Archive Warnings'].unique())
print(unique_warnings)

['No Archive Warnings Apply', 'Major Character Death', 'Creator Chose Not To Use Archive Warnings', 'Graphic Depictions Of Violence']


In [158]:
# save df_fandom to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/warnings_df_exploded.csv'
save_df_to_csv(path_to_save, df_warnings)

## clean df_additional_tags

In [159]:
df_additional_tags.info()

<class 'pandas.DataFrame'>
Index: 2273 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             2273 non-null   int64 
 1   Rating            2273 non-null   object
 2   Archive Warnings  2273 non-null   object
 3   Category          2273 non-null   object
 4   Fandom            2273 non-null   object
 5   Characters        2273 non-null   object
 6   Relationships     2273 non-null   object
 7   Additional Tags   2273 non-null   str   
 8   Language          2273 non-null   object
 9   Date Published    2273 non-null   object
 10  Date Updated      2273 non-null   object
 11  Word Count        2273 non-null   int64 
 12  Chapter Count     2273 non-null   object
 13  Comment Count     2273 non-null   int64 
 14  Kudos Count       2273 non-null   int64 
 15  Bookmarks Count   2273 non-null   int64 
 16  Hits Count        2273 non-null   int64 
dtypes: int64(6), object(10), str(1)

In [160]:
display(df_additional_tags)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,Team as Family,English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,Team Dynamics,English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,Affectionate Insults,English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,Veterans,English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,Pre-Canon,English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James",the romanticism of holding your coat over some...,English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...",missing interlude,English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...",Blink and you'll miss it,English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...",but the deserve to be soft,English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [161]:
unique_additional = list(df_additional_tags['Additional Tags'].unique())
print(unique_additional)

['Team as Family', 'Team Dynamics', 'Affectionate Insults', 'Veterans', 'Pre-Canon', 'Implied/Referenced Alcohol Abuse/Alcoholism', 'Slice of Life', 'Dialogue Heavy', 'Why Did I Write This?', 'someone give this girl a hug', 'The plot of this circle I described as three men and a baby', 'Big Brother Malcolm Trills', 'God I love them already', 'this cant be the first crimson mirror fic on here', 'Fluff and Angst', 'Angst', 'Hurt', 'Hurt No Comfort', 'Hurt/Comfort', 'Emotional Hurt/Comfort', 'mentions of war and the horrible things connected to it', 'Horror', 'Body Horror', 'Eldritch', 'Trauma', 'Death', 'Probably no happy end', 'Dead Dove: Do Not Eat', 'Self-Harm', 'Character Death', 'Murder Mystery', 'Arson', 'Disability', 'Canon Disabled Character', 'Bugs & Insects', 'This is a Rajan fic the bugs are a big part', 'Drug Use', 'Implied/Referenced Drug Use', 'Canon-Typical Violence', 'Masturbation', 'Gambling', 'Blood and Gore', 'Minor Character Death', 'Implied/Referenced Character Death

In [162]:
# save df_fandom to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/additional_tags_df_exploded.csv'
save_df_to_csv(path_to_save, df_additional_tags)

## clean df_categories

In [163]:
df_categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   index             211 non-null    int64 
 1   Rating            211 non-null    object
 2   Archive Warnings  211 non-null    object
 3   Category          211 non-null    str   
 4   Fandom            211 non-null    object
 5   Characters        211 non-null    object
 6   Relationships     211 non-null    object
 7   Additional Tags   211 non-null    object
 8   Language          211 non-null    object
 9   Date Published    211 non-null    object
 10  Date Updated      211 non-null    object
 11  Word Count        211 non-null    int64 
 12  Chapter Count     211 non-null    object
 13  Comment Count     211 non-null    int64 
 14  Kudos Count       211 non-null    int64 
 15  Bookmarks Count   211 non-null    int64 
 16  Hits Count        211 non-null    int64 
dtypes: int64(6), object(10), st

In [164]:
display(df_categories)

,index,Rating,Archive Warnings,Category,Fandom,Characters,Relationships,Additional Tags,Language,Date Published,Date Updated,Word Count,Chapter Count,Comment Count,Kudos Count,Bookmarks Count,Hits Count
0,1,Teen And Up Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Malcolm Trills, Leo Amicus, Grimoria (Candela...",Leo Amicus & Malcolm Trills,"[Team as Family, Team Dynamics, Affectionate I...",English,2024-03-07 00:00:00,2024-03-07 00:00:00,2506,1/1,8,61,4,371
1,2,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),"[Grimoria, Malcolm Trills, Decklan Murphy (Men...",Grimoria & Malcolm Trills,"[Pre-Canon, someone give this girl a hug, The ...",English,2024-03-02 00:00:00,2024-03-02 00:00:00,1067,1/1,3,36,1,290
2,3,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
3,4,Explicit,"[Graphic Depictions Of Violence, Major Charact...",F/M,Candela Obscura (Critical Role Web Series),"[Sean Finnerty (Candela Obscura), Marion Collo...",[Sean Finnerty (Candela Obscura) & Original Ch...,"[Angst, Hurt, Hurt No Comfort, Hurt/Comfort, E...",English,Unknown,2024-02-27 00:00:00,8013,None,3,10,3,160
4,5,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Elsie Roberts (Candela Obscura), Oscar Grimm,...","[Rajan Savarimuthu/Original character, Former ...","[Murder Mystery, Arson, Disability, Canon Disa...",English,2024-02-25 00:00:00,2024-02-25 00:00:00,46890,28/28,28,7,2,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,7,General Audiences,No Archive Warnings Apply,Gen,Candela Obscura (Critical Role Web Series),[Arlo Black (Candela Obscura)],No relationships tagged,[Character Study],English,2023-07-04 00:00:00,2023-07-04 00:00:00,2129,1/1,5,53,7,310
207,8,Explicit,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August James (C...",August 'Auggie' James/ Arlo Black,[Ftm August James],English,Unknown,2023-07-03 00:00:00,6125,None,9,83,8,1259
208,9,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), August ""Auggie""...","Arlo Black/August ""Auggie"" James","[Written post episode 2, First Kiss, Idk they ...",English,2023-07-02 00:00:00,2023-07-02 00:00:00,3378,1/1,48,332,49,1670
209,10,Teen And Up Audiences,No Archive Warnings Apply,F/M,Candela Obscura (Critical Role Web Series),"[Arlo Black (Candela Obscura), Howard Margrove...","[Arlo Black & Howard Margrove, Arlo Black/Howa...","[missing interlude, Blink and you'll miss it, ...",English,2023-06-26 00:00:00,2023-06-26 00:00:00,1082,1/1,2,27,4,220


In [165]:
unique_categories = list(df_categories['Category'].unique())
print(unique_categories)

['Gen', 'F/M', 'M/M', 'No category', 'Multi', 'No categories tagged', 'Gen, Other', 'F/M, M/M, Multi', 'F/F', 'F/M, Multi', 'F/F, F/M, Multi']


In [166]:
# replace text
category_text_replace = {'F/M, M/M, Multi':'Multiple Categories',
                       'F/M, Multi':'Multiple Categories',
                       'Gen, Other':'Multiple Categories',
                       'F/F, F/M, Multi':'Multiple Categories',
                       'No category':'No categories tagged',
                         'Multiple Characters':'Multiple Categories'
                       }

for key in category_text_replace.keys():
    df_categories['Category'] = df_categories['Category'].replace(key, category_text_replace[key])

In [167]:
unique_categories = list(df_categories['Category'].unique())
print(unique_categories)

['Gen', 'F/M', 'M/M', 'No categories tagged', 'Multi', 'Multiple Categories', 'F/F']


In [168]:
# save df_fandom to csv
path_to_save = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/csv_files/categories_df_exploded.csv'
save_df_to_csv(path_to_save, df_categories)